In [1]:
!rm -rf /root/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [10]:
def solve_problem_debug(question, active_model):
    # This prompt format must match exactly what you used during fine-tuning
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = active_model.generate(**inputs, max_new_tokens=200)

    # Decodes and prints the full output so we can see what's happening
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text

# Test it on one question from your dataset
sample_q = test_df.iloc[0]['question']
print("--- MODEL RAW GENERATION ---")
print(solve_problem_debug(sample_q, model))

--- MODEL RAW GENERATION ---
There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas? To find out how big each group of bananas is, we need to divide the total number of bananas by the number of groups they are organized into.

Philip has 290 bananas, and they are organized into 2 groups. So, we divide 290 by 2:

290 bananas ÷ 2 groups = 145 bananas per group

Each group of bananas will have 145 bananas.


In [4]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "./phi3-neuro-symbolic-adapter"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig, AutoModelForCausalLM
import torch

config = AutoConfig.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=False,
    force_download=True,
    revision="main"
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    config=config,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager",
    revision="main"
)

In [7]:
import os
import zipfile
if not os.path.exists(adapter_path):
    if os.path.exists(adapter_path + ".zip"):
        print(f"Unzipping {adapter_path}.zip to {adapter_path}...")
        with zipfile.ZipFile(adapter_path + ".zip", 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(adapter_path))
    else:
        raise FileNotFoundError(f"Adapter path or zip not found: {adapter_path} or {adapter_path}.zip")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Unzipping ./phi3-neuro-symbolic-adapter.zip to ./phi3-neuro-symbolic-adapter...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [11]:
def solve_problem(question, active_model):
    # This prompt asks the model to output ONLY the final numerical answer,
    # which will make it much easier for the evaluation script to handle.
    prompt = f"<|user|>\n{question}\n\nProvide only the final numerical answer as a number.<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = active_model.generate(**inputs, max_new_tokens=50)

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # This simple regex looks for any number in the output
    match = re.search(r'(\d+)', full_text)
    if match:
        return match.group(1)
    return None

In [12]:
import pandas as pd
import torch
import time
import re

def extract_code(text):
    match = re.search(r'```python\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None

def run_evaluation(test_df, active_model, model_name):
    print(f"\nRUNNING EVALUATION: {model_name}")
    print("Symbolic Sandbox & Agentic Loop: OFF")
    correct = 0
    total = len(test_df)

    for index, row in test_df.iterrows():
        target_answer = row['answer']
        agent_code = solve_problem(row['question'], active_model)
        is_correct = False
        if agent_code is not None and str(target_answer) in str(agent_code):
            is_correct = True
            correct += 1
        else:
            print(f"\n--- FAILURE ON Q{index} ---")
            print(f"Question: {row['question']}")
            print(f"Target: {target_answer}")
            print(f"Raw Output: {agent_code}")
            print("---------------------------")

    accuracy = (correct / total) * 100
    print(f"\nFINAL ACCURACY: {accuracy:.2f}% ({correct}/{total})")
    return accuracy

test_df = pd.read_csv("unified_svamp_test.csv").head(50)
run_acc = run_evaluation(test_df, model, "Fine-Tuned Model")


RUNNING EVALUATION: Fine-Tuned Model
Symbolic Sandbox & Agentic Loop: OFF


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- FAILURE ON Q0 ---
Question: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas?
Target: 145
Raw Output: 87
---------------------------

--- FAILURE ON Q1 ---
Question: Marco and his dad went strawberry picking. Marco's dad's strawberries weighed 11 pounds. If together their strawberries weighed 30 pounds. How much did Marco's strawberries weigh?
Target: 19
Raw Output: 11
---------------------------

--- FAILURE ON Q2 ---
Question: Edward spent $ 6 to buy 2 books each book costing him the same amount of money. Now he has $ 12. How much did each book cost?
Target: 3
Raw Output: 6
---------------------------

--- FAILURE ON Q3 ---
Question: Frank was reading through his favorite book. The book had 3 chapters, each with the same number of pages. It has a total of 594 pages. It took Frank 607 days to finish the book. How many pages are in each chapter?
Target: 